# Notatnik do przekształcania cech i danych w sposób akceptowalny dla modeli 

## Załadowanie bibliotek

In [1]:
import pandas as pd 
import numpy as np

## Załadowanie danych

In [2]:
df = pd.read_csv('../data/processed/df_merged.csv', index_col= 0)

In [4]:
print(df.shape)
print(df.columns.to_list())
print(df.info())

(413169, 12)
['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge', 'Brand', 'Gas', 'Region', 'Density', 'TotalClaimAmount', 'ClaimCount']
<class 'pandas.core.frame.DataFrame'>
Index: 413169 entries, 0 to 413168
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   PolicyID          413169 non-null  int64  
 1   ClaimNb           413169 non-null  int64  
 2   Exposure          413169 non-null  float64
 3   Power             413169 non-null  object 
 4   CarAge            413169 non-null  int64  
 5   DriverAge         413169 non-null  int64  
 6   Brand             413169 non-null  object 
 7   Gas               413169 non-null  object 
 8   Region            413169 non-null  object 
 9   Density           413169 non-null  int64  
 10  TotalClaimAmount  413169 non-null  float64
 11  ClaimCount        413169 non-null  float64
dtypes: float64(3), int64(5), object(4)
memory usage: 41.0+ MB
None


In [6]:
diff_mask = df["ClaimNb"] != df["ClaimCount"]

print("Liczba różniących się wierszy:", diff_mask.sum())

print(
    df.loc[
        diff_mask,
        ["PolicyID", "ClaimNb", "ClaimCount"]
    ].head(20)
)

Liczba różniących się wierszy: 0
Empty DataFrame
Columns: [PolicyID, ClaimNb, ClaimCount]
Index: []


usunięcie niepotrzebnej kolumny

In [8]:
df = df.drop(columns=["ClaimCount"])

## Tworzenie ramek danych dla modeli 

In [24]:
df_model = df.copy()

# Cechy wykorzystywane w modelach
feature_cols = [
    "DriverAge",
    "CarAge",
    "Power",
    "Brand",
    "Gas",
    "Region",
    "Density",
]

targets = {
    'frequency' : 'Frequency', 
    'severity' : 'Severity', 
    'pure_premium' : 'PurePremium'
}

### Częstość

In [13]:
df_frequency = df_model.drop(columns = ['TotalClaimAmount']).copy()
print(df_frequency.columns.to_list())

['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge', 'Brand', 'Gas', 'Region', 'Density']


### Wysokość szkody

In [16]:
df_severity = df_model.loc[
    df_model['ClaimNb'] > 0 
    & 
    df_model['TotalClaimAmount'].notna()
].copy()

df_severity['Severity'] = df_severity['TotalClaimAmount'] / df_severity['ClaimNb']

print(df_severity.columns.to_list())

df_severity.head()

['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge', 'Brand', 'Gas', 'Region', 'Density', 'TotalClaimAmount', 'Severity']


,PolicyID,ClaimNb,Exposure,Power,CarAge,DriverAge,Brand,Gas,Region,Density,TotalClaimAmount,Severity
32,33,1,0.75,g,1,61,Japanese (except Nissan) or Korean,Regular,Ile-de-France,27000,302.0,302.0
40,41,1,0.14,l,5,50,Japanese (except Nissan) or Korean,Diesel,Basse-Normandie,56,2001.0,2001.0
91,92,1,0.14,d,0,36,Japanese (except Nissan) or Korean,Regular,Ile-de-France,4792,1449.0,1449.0
95,96,2,0.62,j,0,51,Japanese (except Nissan) or Korean,Regular,Ile-de-France,27000,10870.0,5435.0
141,142,1,0.75,e,0,34,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,1565,1390.0,1390.0


### Pure premium

In [20]:
df_pure_premium = df_model.loc[
    df_model['Exposure'] > 0 
    & 
    df_model['TotalClaimAmount'].notna()
].copy()

df_pure_premium['PurePremium'] = df_pure_premium['TotalClaimAmount'] / df_pure_premium['Exposure']

print(df_pure_premium.columns.to_list())
df_pure_premium.head()
df_pure_premium['PurePremium'].describe()

['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge', 'Brand', 'Gas', 'Region', 'Density', 'TotalClaimAmount', 'PurePremium']


count    4.131690e+05
mean     3.711453e+02
std      3.493029e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.824670e+07
Name: PurePremium, dtype: float64

10 obserwacji z najwiekszym pure premium

In [ ]:
df_pure_premium.nlargest(
    10, "PurePremium"
)[
    [
        "PolicyID",
        "ClaimNb",
        "Exposure",
        "TotalClaimAmount",
        "PurePremium",
    ]
]

,PolicyID,ClaimNb,Exposure,TotalClaimAmount,PurePremium
18580,18581,1,0.010000,182467.0,1.824670e+07
101090,101091,1,0.220000,2036833.0,9.258332e+06
294454,294455,1,0.008219,31931.0,3.884938e+06
35317,35318,2,0.430000,1403452.0,3.263842e+06
245857,245858,1,0.010000,29921.0,2.992100e+06
378779,378780,2,0.005479,14569.0,2.658843e+06
167350,167351,1,0.005479,11892.0,2.170290e+06
67109,67110,1,0.060000,120931.0,2.015517e+06
24969,24970,1,0.040000,72960.0,1.824000e+06
182672,182673,1,0.040000,69983.0,1.749575e+06
